<a href="https://colab.research.google.com/github/ilincabaiasu/IB9AU/blob/main/Task_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Required Task 16

### What I found interesting:

- Being able to go through the agent's thinking process
- The agent is active, decides by itself what it needs to do




---



### Details:

Add a new tool called get_pe_ratio that fetches the Price-to-Earnings ratio. Then ask
the agent if Apple is 'overvalued' compared to the average market P/E of 25. Check the python notebook Agents1.ipynb for context and further details.

Hint:
stock = yf.Ticker(ticker)

PE ratio is often in 'summaryDetail' or 'info'

pe = stock.info.get('trailingPE', 'N/A')



---



In [1]:
!pip install -q -U google-generativeai
!pip install -q yfinance

import google.generativeai as genai
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("✅ API configured.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ API configured.


### Define the 3 Tools

In [2]:
import yfinance as yf

# ── Original Tool 1 ───────────────────────────────────────────────────────────
def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"


# ── Original Tool 2 ───────────────────────────────────────────────────────────
def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)
        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"
        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"


# ── NEW Tool 3: P/E Ratio ─────────────────────────────────────────────────────
def get_pe_ratio(ticker: str):
    """
    Fetches the trailing Price-to-Earnings (P/E) ratio for a given stock.
    The P/E ratio measures how expensive a stock is relative to its earnings.
    A P/E above the market average (~25) may indicate the stock is overvalued.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', None)

        if pe is None:
            return {"ticker": ticker, "pe_ratio": "N/A", "assessment": "P/E data unavailable"}

        pe = round(pe, 2)

        # Compare against market average of 25
        MARKET_AVERAGE_PE = 25
        if pe > MARKET_AVERAGE_PE * 1.5:      # more than 50% above market avg
            assessment = f"Significantly Overvalued (P/E {pe} vs market avg {MARKET_AVERAGE_PE})"
        elif pe > MARKET_AVERAGE_PE:           # above market avg
            assessment = f"Potentially Overvalued (P/E {pe} vs market avg {MARKET_AVERAGE_PE})"
        elif pe < MARKET_AVERAGE_PE * 0.7:    # more than 30% below market avg
            assessment = f"Potentially Undervalued (P/E {pe} vs market avg {MARKET_AVERAGE_PE})"
        else:
            assessment = f"Fairly Valued (P/E {pe} vs market avg {MARKET_AVERAGE_PE})"

        return {
            "ticker": ticker,
            "pe_ratio": pe,
            "market_average_pe": MARKET_AVERAGE_PE,
            "assessment": assessment
        }
    except Exception as e:
        return {"ticker": ticker, "pe_ratio": "N/A", "assessment": f"Error: {e}"}


print("✅ All three tools defined: get_stock_price, get_company_risk_score, get_pe_ratio")

✅ All three tools defined: get_stock_price, get_company_risk_score, get_pe_ratio


### Initialise Agent with All 3 Tools

In [3]:
# All three tools registered — the agent can now call any of them
tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Agent initialised with 3 tools.")

✅ Agent initialised with 3 tools.


### Is Apple Overvalued?

In [4]:
query = (
    "Fetch Apple's current P/E ratio and tell me whether it is overvalued "
    "compared to the average market P/E of 25. "
    "Also provide its current stock price and risk assessment to give a complete picture."
)

print(f"❓ USER QUERY: {query}\n")
response = chat.send_message(query)

print("\n🤖 AGENT RESPONSE:")
print(response.text)

❓ USER QUERY: Fetch Apple's current P/E ratio and tell me whether it is overvalued compared to the average market P/E of 25. Also provide its current stock price and risk assessment to give a complete picture.

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...
  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for AAPL ...
  ... ⚠️ TOOL CALL: Fetching Risk Metrics for AAPL ...

🤖 AGENT RESPONSE:
Apple's current P/E ratio is 31.35, which suggests it is potentially overvalued compared to the average market P/E of 25. Its current stock price is $247.99. The company's risk assessment is "Moderate Risk" with a beta of 1.116, indicating slightly higher volatility than the overall market.


### Inspect Agent's Thought Process

In [5]:
print("🕵️ HISTORY INSPECTION — ReAct loop trace:\n")
for message in chat.history:
    role = message.role
    print(f"--- {role.upper()} ---")
    for part in message.parts:
        if fn := part.function_call:
            print(f"🔧 ACTION    : Called '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"📨 OBSERVATION: Tool returned: {resp.response}")
        else:
            print(part.text)
    print()

🕵️ HISTORY INSPECTION — ReAct loop trace:

--- USER ---
Fetch Apple's current P/E ratio and tell me whether it is overvalued compared to the average market P/E of 25. Also provide its current stock price and risk assessment to give a complete picture.

--- MODEL ---
🔧 ACTION    : Called 'get_pe_ratio' with args: {'ticker': 'AAPL'}

--- USER ---
📨 OBSERVATION: Tool returned: <proto.marshal.collections.maps.MapComposite object at 0x7d429d8a6480>

--- MODEL ---
🔧 ACTION    : Called 'get_stock_price' with args: {'ticker': 'AAPL'}

--- USER ---
📨 OBSERVATION: Tool returned: <proto.marshal.collections.maps.MapComposite object at 0x7d429d8a5d30>

--- MODEL ---
🔧 ACTION    : Called 'get_company_risk_score' with args: {'ticker': 'AAPL'}

--- USER ---
📨 OBSERVATION: Tool returned: <proto.marshal.collections.maps.MapComposite object at 0x7d429d8a77d0>

--- MODEL ---
Apple's current P/E ratio is 31.35, which suggests it is potentially overvalued compared to the average market P/E of 25. Its curren